In [1]:
import spacy
from spacy.matcher import PhraseMatcher, Matcher
from spacy.language import Language
from spacy.tokens import Span
from spacy.util import filter_spans
import re
import pandas as pd
import joblib



c:\Users\My-PC\OneDrive\Desktop\medical-chatbot\env\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


In [2]:
nlp=spacy.load('en_core_web_md')
matcher_0 = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher_1 = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher_2 = Matcher(nlp.vocab)
matcher_3 = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher_4 = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher_5 = Matcher(nlp.vocab)


In [3]:
df=pd.read_csv('../saved_files/nlp_dataset/A_Z_medicines_dataset_of_India.csv')

df.head()

,id,name,price(₹),Is_discontinued,manufacturer_name,type,pack_size_label,short_composition1,short_composition2
0,1,Augmentin 625 Duo Tablet,223.42,False,Glaxo SmithKline Pharmaceuticals Ltd,allopathy,strip of 10 tablets,Amoxycillin (500mg),Clavulanic Acid (125mg)
1,2,Azithral 500 Tablet,132.36,False,Alembic Pharmaceuticals Ltd,allopathy,strip of 5 tablets,Azithromycin (500mg),NaN
2,3,Ascoril LS Syrup,118.00,False,Glenmark Pharmaceuticals Ltd,allopathy,bottle of 100 ml Syrup,Ambroxol (30mg/5ml),Levosalbutamol (1mg/5ml)
3,4,Allegra 120mg Tablet,218.81,False,Sanofi India Ltd,allopathy,strip of 10 tablets,Fexofenadine (120mg),NaN
4,5,Avil 25 Tablet,10.96,False,Sanofi India Ltd,allopathy,strip of 15 tablets,Pheniramine (25mg),NaN


In [4]:
list1 = df['short_composition1'].dropna().unique().tolist()
list2 = df['short_composition2'].dropna().unique().tolist()


drug_list = list1+list2

print('drugs name:')
print(drug_list[:5])

print('total nunmber of drugs:',len(drug_list))



drugs name:
['Amoxycillin  (500mg) ', 'Azithromycin (500mg)', 'Ambroxol (30mg/5ml) ', 'Fexofenadine (120mg)', 'Pheniramine (25mg)']
total nunmber of drugs: 11503


In [5]:
pattern=r"\s*\(.*?\)"

for i in range(len(drug_list)):
    drug_list[i] = re.sub(pattern,"", drug_list[i].strip())

print(drug_list[:5])

['Amoxycillin', 'Azithromycin', 'Ambroxol', 'Fexofenadine', 'Pheniramine']


In [6]:
patterns = [nlp.make_doc(med) for med in drug_list]

matcher_0.add("DRUGS", patterns)

In [7]:
df_medicine_list= df['name'].dropna().unique().tolist()

raw_medicine_list= [ i.split() for i in df_medicine_list ]

raw_medicine_list[:5]

[['Augmentin', '625', 'Duo', 'Tablet'],
 ['Azithral', '500', 'Tablet'],
 ['Ascoril', 'LS', 'Syrup'],
 ['Allegra', '120mg', 'Tablet'],
 ['Avil', '25', 'Tablet']]

In [8]:
          
medicine_list= list(set(map(lambda x : ' '.join(x[:2]),raw_medicine_list)))


print('medicine name:')
print(medicine_list[10:30])

print('total medicine:',len(medicine_list))


medicine name:
['Eljoint 50mg', 'Uddiol 150mg', 'Chlormit Tablet', 'Lesmezole Capsule', 'Kufris AM', 'Combunex 800', 'Nexiflow D', 'Stansunate 60mg', 'Winacin 500mg', 'Nausidome 1mg/ml', 'Ketol-Z Soap', 'Nanoxel 300mg', 'Carelol 3.125mg', 'Itroze 200mg', 'Irap 100mg', 'E-Fen 50mg/500mg', 'Rinomune Forte', 'Orthogesic 100mg', 'Migravin 500', 'Tmr H']
total medicine: 232016


In [9]:
patterns = [nlp.make_doc(med) for med in medicine_list]

matcher_1.add("MEDICINE", patterns)

In [10]:
pattern = [
    {"LIKE_NUM": True},
    {"LOWER": {"IN": ["mg", "ml","g","mcg"]}}
]

matcher_2.add("DOSAGE", [pattern])

In [11]:
doc = nlp("Doctor gave me paracetamol 500mg and Dolo 650 for fever")

print("PhraseMatcher Results:")
for match_id, start, end in matcher_0(doc):
    print("drug:", doc[start:end].text)

print("\nPhraseMatcher Results:")
for match_id, start, end in matcher_1(doc):
    print("Medicine:", doc[start:end].text)

print("\nMatcher Results:")
for match_id, start, end in matcher_2(doc):
    print("Dosage:", doc[start:end].text)

PhraseMatcher Results:
drug: paracetamol

PhraseMatcher Results:
Medicine: Dolo 650

Matcher Results:
Dosage: 500mg


In [12]:
disease_dataset = pd.read_csv('../saved_files/nlp_dataset/Disease and symptoms dataset.csv')

disease_name = disease_dataset['diseases'].dropna().unique().tolist()


In [13]:
disease_list = list(map(lambda x: x.lower(),disease_name)) 
print('disease name:')
print(disease_list[:5])
print('total disease:',len(disease_list))

disease name:
['panic disorder', 'vocal cord polyp', 'turner syndrome', 'cryptorchidism', 'poisoning due to ethylene glycol']
total disease: 773


In [14]:
patterns = [nlp.make_doc(med) for med in disease_list]

matcher_3.add("DISEASES", patterns)

In [15]:
add_common_disease = pd.read_csv('../saved_files/nlp_dataset/symptom_dataset.csv')  

common_disease = add_common_disease['Symptom_Name'].tolist()



In [16]:
patterns = [nlp.make_doc(med) for med in common_disease]

matcher_4.add("SYMPTOM", patterns)

In [ ]:
print('symptoms name:')

print(common_disease[:5])

print('total symptoms:',len(common_disease))

symptoms name:
   Symptom_ID      Symptom_Name
0           1             Fever
1           2        High fever
2           3        Mild fever
3           4   Low-grade fever
4           5  Continuous fever
total symptoms: 229


In [18]:
pattern_1 = [
    {"LIKE_NUM": True},
    {"LOWER": {"IN": ["day", "days", "week", "weeks", "month", "months"]}}
]

pattern_2 = [
    {"LOWER": {"IN": ["yesterday", "today", "tomorrow"]}}
]

pattern_3 = [
    {"LOWER": "day"},
    {"LOWER": "before"},
    {"LOWER": "yesterday"}
]

pattern_4 = [
    {"LOWER": "last"},
    {"LOWER": {'IN': ['day','month','week','year','morning','afternoon','evening','night']}},

]

matcher_5.add("DURATION", [pattern_1])
matcher_5.add("RELATIVE_DATE", [pattern_2, pattern_3,pattern_4])

In [19]:
matchers = [
    (matcher_0, "DRUG"),
    (matcher_1, "MEDICINE"),
    (matcher_2, "DOSAGE"),
    (matcher_4, "SYMPTOM"),
    (matcher_3, "DISEASE"),
    (matcher_5, "TIME_EXPRESSION"),
]

In [20]:
@Language.component("medical_matcher")
def medicine_words(doc):

    new_ents = list(doc.ents)

    for matcher, label in matchers:
        for _, start, end in matcher(doc):

            span = Span(doc, start, end, label=label)

            new_ents = [
                e for e in new_ents
                if not (e.start < end and e.end > start)
            ]

            new_ents.append(span)

    doc.ents = filter_spans(new_ents)
    return doc

In [21]:
nlp.add_pipe("medical_matcher", after="ner")

print(nlp.pipe_names)

['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner', 'medical_matcher']


In [22]:
import dateparser

In [23]:
# doc = nlp("Doctor gave me paracetamol 500mg and Dolo 650 for fever FOR TOMORROW")
# doc=nlp('my sister are feeling pain in legs for 3 days')
# doc=nlp('my sister have cold sore')
# doc=nlp('i am feeling severe pain in legs for 3 days')

def content_details(doc):
    trigger_verbs = ["have",'give', 'take', "suffer", "experience", "diagnose", "feel",'be']

    severity_map = {
    "severe": ["severe", "extreme", "intense", "unbearable", "very bad"],
    "moderate": ["moderate", "average", "normal pain"],
    "mild": ["mild", "slight", "little", "light"]
    }
    
    self_terms = {"i", "me","mine"}

    family_terms = {
        "mother","father","mom","dad","brother","sister",
        "uncle","aunt","grandmother","grandfather",
        "son","daughter","wife","husband","child","baby","aunt"
    }

    friend_terms = {
        "friend","buddy","colleague","roommate",
        "classmate","neighbor"
    }


    def identify_subject(text):
        doc = nlp(text)

        for token in doc:
            word = token.text.lower()

            if word in family_terms:
                return word
            elif word in friend_terms:
                return "friends"
            elif word in self_terms:
                return "I"
    
    def get_severity(text):
        text = text.lower()

        for level, words in severity_map.items():
            for word in words:
                if word in text:
                    return level

        return None

    detail_extract={'patient': None,
    'severity':None,
    'condition':None,
    'disease':None,
    'medicine':None,
    'duration':None,
    'timestamp':None
    }

    detail_extract['severity']=get_severity(doc.text)
    
    detail_extract['patient']= identify_subject(doc)

    for ent in doc.ents:
        if ent.label_ == "MEDICINE":
            detail_extract['medicine'] = ent.text 
        if ent.label_ == "DISEASE":
            detail_extract['disease'] = ent.text
        if ent.label_ == "SYMPTOM":
            detail_extract['condition'] = ent.text
        if ent.label_ == "TIME_EXPRESSION" or ent.label_ == 'DATE':
            detail_extract['duration'] = ent.text
                        

                    
    if(detail_extract['duration']):
        parsed_date = dateparser.parse(detail_extract['duration'])
        try:
            formatted_date = parsed_date.strftime("%d/%m/%Y")
            detail_extract['timestamp']=formatted_date
        except:
            detail_extract['timestamp'] = None

    return detail_extract





In [24]:
disease_dataset[disease_dataset['diseases'].str.contains('ulcer', case=False)]

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
127604,decubitus ulcer,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
127605,decubitus ulcer,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
127606,decubitus ulcer,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
127607,decubitus ulcer,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
127608,decubitus ulcer,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242346,gastroduodenal ulcer,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
242347,gastroduodenal ulcer,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
242348,gastroduodenal ulcer,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
242349,gastroduodenal ulcer,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
df[df['name'].str.contains('Omalirel', case=False)]

,id,name,price(₹),Is_discontinued,manufacturer_name,type,pack_size_label,short_composition1,short_composition2
162720,162721,Omalirel Injection,13145.0,False,Reliance Life Sciences,allopathy,vial of 1 Injection,Omalizumab (150mg),NaN


In [26]:
add_common_disease[add_common_disease['Symptom_Name'].str.contains('ulcer', case=False)]

,Symptom_ID,Symptom_Name
135,136,Mouth ulcer
227,228,ulcer
228,229,stomach ulcer


In [27]:
demo=[
    "I have had a severe headache for the past 3 days.",

"My stomach has been hurting since last night.",

"I feel dizzy whenever I stand up.",

"I have chest pain and shortness of breath.",

"My child has a fever of 102°F since yesterday.",

"I am experiencing nausea and vomiting after eating.",

"There is swelling and redness around my ankle.",

"I have dry cough and mild fever.",

"My throat is sore and I can't swallow properly.",

"I have frequent urination and burning sensation.",

'i am suffering from fever',

]

In [28]:
for word in demo:
    doc=nlp(word)
    print(doc)
    print(content_details(doc),end='\n\n')

I have had a severe headache for the past 3 days.
{'patient': 'I', 'severity': 'severe', 'condition': 'headache', 'disease': None, 'medicine': None, 'duration': '3 days', 'timestamp': '09/03/2026'}

My stomach has been hurting since last night.
{'patient': None, 'severity': None, 'condition': None, 'disease': None, 'medicine': None, 'duration': 'last night', 'timestamp': None}

I feel dizzy whenever I stand up.
{'patient': 'I', 'severity': None, 'condition': 'dizzy', 'disease': None, 'medicine': 'I feel', 'duration': None, 'timestamp': None}

I have chest pain and shortness of breath.
{'patient': 'I', 'severity': None, 'condition': 'shortness of breath', 'disease': None, 'medicine': None, 'duration': None, 'timestamp': None}

My child has a fever of 102°F since yesterday.
{'patient': 'child', 'severity': None, 'condition': 'fever', 'disease': None, 'medicine': None, 'duration': 'yesterday', 'timestamp': '11/03/2026'}

I am experiencing nausea and vomiting after eating.
{'patient': 'I',

In [29]:
with open('demo_nlp_text.txt','+r') as file:
    result= file.read()


pattern=r'[^A-Za-z0-9 \n \t]'

filter_result=re.sub(pattern,'',result)
print(filter_result)

SRI DHANVANTHRY
MEDICALS
1102 SOUTH MAIN STREET MAIN ROAD TAPANASAM
DLNOTUT214820 TUT214820GSTIN33AAPM3480A12D
Ph2243572 7639554605
CASH Bill No 61678
Date 011125
Time 185504
Dr PKRISHNAMURTHY MD
Name RAVI
Address

QTY MRP PARTICULARS RACK MFR BATCH EXP AMOUNT
10 29966 RENERVE PLUS CAP 15 MM GRAN GREL2509 0627 19977
TOTSGST 476
TOTCGST 476
TOTCOST 476
Total 19976

ID RAJAGOPAL
Goods once sold cannot be taken back
E  OE
Phar Sign
VRAMKUMAR
WISH YOU SPEEDY RECOVERY

Sub Total 19976
Discount 0 
Total 19900


In [33]:
doc=nlp(filter_result)

for ent in doc.ents:
    print(ent.text," ",ent.label_ )
    # # print(ent.text," ", ent.label_)
    # if(ent.label_=='DOSAGE' or ent.label_=='MEDICINE'):
    #     print(ent.text," ",ent.label_)

MEDICALS   ORG
1102   CARDINAL
7639554605   DATE
PKRISHNAMURTHY   ORG
RAVI
Address   ORG
EXP   CARDINAL
10 29966   CARDINAL
RENERVE PLUS   MEDICINE
15 MM GRAN GREL2509   QUANTITY
19977   DATE
476   CARDINAL
476   CARDINAL
476   CARDINAL
19976   DATE
OE   DATE
Phar Sign
VRAMKUMAR   ORG
19976   DATE
0   CARDINAL


In [31]:
doc=nlp("Take paracetamol 500mg twice daily")

for i, ent in enumerate(doc.ents):
    if ent.label_ == "MEDICINE" or ent.label_ == "DRUG":
        if i + 1 < len(doc.ents):
            next_ent = doc.ents[i + 1]
            if next_ent.label_ == "DOSAGE":
                print(ent.text, "->", next_ent.text)

paracetamol -> 500mg


In [ ]:
nlp.to_disk('../saved_files/medical_nlp_model')

['../saved_files/NLP_MODEL.pkl']